# 06 — LightGBM (modelo principal)

**Objetivo:** Entrenar el modelo principal de forecasting del proyecto usando LightGBM, y compararlo contra el baseline (notebook 05) para demostrar el aporte de un modelo sofisticado.

**Fase CRISP-DM:** 4 — Modelado.

**Inputs:**
- `data/processed/dataset_modelable.parquet` — dataset con 29 features
- Tabla `clasificacion_abc_xyz` (MySQL) — para reportes por segmento
- Tabla `pronosticos` (MySQL) — predicciones del baseline para comparar

**Outputs:**
- Modelo entrenado guardado en `data/processed/lightgbm_model.txt`
- Predicciones en tabla `pronosticos` (modelo='lightgbm')
- Métricas en tabla `metricas_modelos` (modelo='lightgbm')
- Feature importance en `reports/figures/lightgbm/`

**Por qué LightGBM:**

LightGBM es un algoritmo de gradient boosting con árboles de decisión que:
- Maneja series temporales agregadas excelentemente cuando se le dan lags y features de calendario.
- Procesa categóricas nativamente sin one-hot encoding.
- Es rápido: minutos en lugar de horas.
- Es interpretable mediante feature importance.

**Hipótesis a validar:**

> LightGBM, con las 29 features creadas, debe mejorar significativamente al baseline (media móvil 4 semanas), especialmente en los segmentos ABC con alto valor (A, AY, AZ).

**Autor:** Equipo del proyecto — Diego Andrés De Jesús Montenegro y Luis David Andrade Díaz
**Fecha:** mayo 2026

In [ ]:
# =========================
# Imports
# =========================
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from sqlalchemy import text

import lightgbm as lgb
print(f"LightGBM version: {lgb.__version__}")

from src.db import get_engine

# Configuracion visual
COLOR_PRIMARIO = "#1F4E78"
COLOR_SECUNDARIO = "#E67E22"
COLOR_VERDE = "#27AE60"
COLOR_ROJO = "#C0392B"
COLOR_LIGHTGBM = "#9B59B6"  # Morado distintivo para lightgbm

plt.rcParams.update({
    "figure.figsize": (12, 5),
    "figure.dpi": 100,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

FIG_DIR = ROOT / "reports" / "figures" / "lightgbm"
FIG_DIR.mkdir(parents=True, exist_ok=True)

def guardar(fig, nombre):
    out = FIG_DIR / f"{nombre}.png"
    fig.savefig(out, dpi=150, bbox_inches="tight", facecolor="white")
    print(f"  Guardado: {out.relative_to(ROOT)}")

print("Setup completado.")

## 1. Carga del dataset modelable

In [ ]:
ruta_parquet = ROOT / "data" / "processed" / "dataset_modelable.parquet"
dataset = pd.read_parquet(ruta_parquet)
print(f"Dataset: {len(dataset):,} filas x {len(dataset.columns)} columnas")
print(f"\nSplits:")
print(dataset["split"].value_counts())

## 2. Carga de predicciones del baseline (para comparar)

In [ ]:
engine = get_engine()

# Cargar predicciones del baseline desde MySQL
baseline_preds = pd.read_sql(
    """
    SELECT item, centro_operacion, fecha_inicio_semana,
           cantidad_real, cantidad_predicha AS pred_baseline
    FROM pronosticos
    WHERE modelo = 'baseline_ma4'
    """,
    engine,
    parse_dates=["fecha_inicio_semana"],
)
print(f"Predicciones del baseline cargadas: {len(baseline_preds):,} filas")

# Cargar clasificacion ABC/XYZ
clasif = pd.read_sql(
    "SELECT item, clase_abc, segmento_abc_xyz FROM clasificacion_abc_xyz",
    engine,
)
print(f"Clasificacion ABC/XYZ: {len(clasif):,} SKUs")

## 3. Separación de features y target

Definimos qué columnas son features predictivas (X) y cuál es el target (y).

In [ ]:
# Lista de features (29 features del notebook 03)
features = [
    # Lags y tendencia
    "lag_1", "lag_2", "lag_4", "lag_8", "lag_13", "lag_52",
    "rolling_mean_4", "rolling_mean_13", "rolling_std_4", "diff_1",
    # Calendario
    "mes", "trimestre", "dia_anio", "sin_mes", "cos_mes",
    "sin_semana", "cos_semana", "num_festivos",
    # Categoricas codificadas
    "nombre_linea_n1_cod", "nombre_linea_n2_cod",
    "proveedor_codigo_cod", "centro_operacion_cod",
    # Clasificacion
    "clase_abc_num", "clase_xyz_num",
    # Estadisticas por SKU (calculadas sobre train)
    "sku_mean", "sku_median", "sku_std", "sku_min", "sku_max",
]

target = "cantidad_total"

# Features categoricas (LightGBM las trata distinto)
categorical_features = [
    "mes", "trimestre",
    "nombre_linea_n1_cod", "nombre_linea_n2_cod",
    "proveedor_codigo_cod", "centro_operacion_cod",
    "clase_abc_num", "clase_xyz_num",
]

print(f"Total features: {len(features)}")
print(f"Features categoricas: {len(categorical_features)}")

In [ ]:
# Separar splits
train = dataset[dataset["split"] == "train"].copy()
valid = dataset[dataset["split"] == "valid"].copy()
test = dataset[dataset["split"] == "test_2026"].copy()

X_train, y_train = train[features], train[target]
X_valid, y_valid = valid[features], valid[target]
X_test, y_test = test[features], test[target]

print(f"Train:      {len(X_train):>8,} filas")
print(f"Validation: {len(X_valid):>8,} filas")
print(f"Test 2026:  {len(X_test):>8,} filas")

## 4. Hiperparámetros del modelo

Hiperparámetros razonables sin optimizar (justificación documentada).

**Decisiones técnicas:**

- `learning_rate = 0.05`: tasa de aprendizaje moderada, balance entre velocidad y estabilidad.
- `num_leaves = 63`: profundidad razonable. Más hojas → más overfitting, menos hojas → underfit.
- `min_data_in_leaf = 20`: cada hoja debe tener al menos 20 observaciones. Evita ruido.
- `feature_fraction = 0.9`: usa 90% de features en cada árbol (regularización).
- `bagging_fraction = 0.8`: usa 80% de filas en cada árbol.
- `lambda_l2 = 0.1`: regularización L2 ligera.

**Estrategia de parada:**
- Hasta 1000 árboles.
- Early stopping de 50 iteraciones sobre validation (si no mejora, para).

In [ ]:
params = {
    "objective": "regression",
    "metric": "mae",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_data_in_leaf": 20,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "lambda_l2": 0.1,
    "verbose": -1,
    "seed": 42,
}

num_boost_round = 1000
early_stopping_rounds = 50

print("Hiperparametros configurados.")
print(f"Max iteraciones: {num_boost_round}")
print(f"Early stopping: {early_stopping_rounds} rondas")

## 5. Entrenamiento

Convertimos a `lgb.Dataset` (formato nativo de LightGBM) y entrenamos con early stopping sobre validation.

In [ ]:
# Crear datasets de LightGBM
lgb_train = lgb.Dataset(
    X_train, y_train,
    categorical_feature=categorical_features,
)
lgb_valid = lgb.Dataset(
    X_valid, y_valid,
    categorical_feature=categorical_features,
    reference=lgb_train,
)

print("Iniciando entrenamiento...")
print("(Imprime cada 50 iteraciones el MAE en train y validation)\n")

modelo = lgb.train(
    params,
    lgb_train,
    num_boost_round=num_boost_round,
    valid_sets=[lgb_train, lgb_valid],
    valid_names=["train", "valid"],
    callbacks=[
        lgb.early_stopping(stopping_rounds=early_stopping_rounds, verbose=True),
        lgb.log_evaluation(period=50),
    ],
)

print(f"\nMejor iteracion: {modelo.best_iteration}")
print(f"Mejor MAE en validation: {modelo.best_score['valid']['l1']:.4f}")

## 6. Predicciones en validation y test 2026

In [ ]:
# Predicciones
y_pred_valid = modelo.predict(X_valid, num_iteration=modelo.best_iteration)
y_pred_test = modelo.predict(X_test, num_iteration=modelo.best_iteration)

# Aplicar clip a 0 (no hay cantidades negativas)
y_pred_valid = np.clip(y_pred_valid, 0, None)
y_pred_test = np.clip(y_pred_test, 0, None)

print(f"Predicciones validation: {len(y_pred_valid):,}")
print(f"Predicciones test 2026:  {len(y_pred_test):,}")
print(f"\nEstadisticas de predicciones (validation):")
print(pd.Series(y_pred_valid).describe())

## 7. Función de métricas (reutilizada del notebook 05)

In [ ]:
def calcular_metricas(y_real, y_pred, nombre=""):
    """Calcula MAE, RMSE, MAPE y sMAPE."""
    y_real = np.asarray(y_real, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    error = y_real - y_pred
    abs_error = np.abs(error)

    mae = abs_error.mean()
    rmse = np.sqrt((error ** 2).mean())

    mask_no_cero = y_real > 0
    mape = (abs_error[mask_no_cero] / y_real[mask_no_cero]).mean() * 100 if mask_no_cero.sum() > 0 else np.nan

    denominador = (np.abs(y_real) + np.abs(y_pred)) / 2
    mask_smape = denominador > 0
    smape = (abs_error[mask_smape] / denominador[mask_smape]).mean() * 100 if mask_smape.sum() > 0 else np.nan

    metricas = {"n_obs": len(y_real), "mae": mae, "rmse": rmse, "mape": mape, "smape": smape}

    if nombre:
        print(f"{nombre}:")
        print(f"  N observaciones: {metricas['n_obs']:>10,}")
        print(f"  MAE:             {metricas['mae']:>10.4f}")
        print(f"  RMSE:            {metricas['rmse']:>10.4f}")
        print(f"  MAPE:            {metricas['mape']:>10.2f}%")
        print(f"  sMAPE:           {metricas['smape']:>10.2f}%")

    return metricas

metricas_valid = calcular_metricas(y_valid, y_pred_valid, nombre="LIGHTGBM EN VALIDATION")
print()
metricas_test = calcular_metricas(y_test, y_pred_test, nombre="LIGHTGBM EN TEST 2026")

## 8. Comparación lado a lado con el baseline

**La pregunta clave del proyecto: ¿LightGBM mejora al baseline?**

In [ ]:
# Cargar metricas del baseline desde MySQL
metricas_baseline = pd.read_sql(
    """
    SELECT split, mae, rmse, mape, smape, n_obs
    FROM metricas_modelos
    WHERE modelo = 'baseline_ma4' AND segmento = 'global'
    """,
    engine,
)
print("Metricas del baseline (de MySQL):")
print(metricas_baseline)

In [ ]:
# Tabla comparativa
print("=" * 75)
print("COMPARACION: BASELINE (MA-4) vs LIGHTGBM")
print("=" * 75)
print(f"{'Metrica':<10} {'Baseline':>15} {'LightGBM':>15} {'Mejora':>15} {'%':>10}")
print("-" * 75)

base_v = metricas_baseline[metricas_baseline["split"] == "valid"].iloc[0]
base_t = metricas_baseline[metricas_baseline["split"] == "test_2026"].iloc[0]

print("\nVALIDATION (Q4 2025):")
for m in ["mae", "rmse", "mape", "smape"]:
    baseline_val = float(base_v[m])
    lgbm_val = metricas_valid[m]
    mejora_abs = baseline_val - lgbm_val
    mejora_pct = (mejora_abs / baseline_val * 100) if baseline_val != 0 else 0
    sufijo = "%" if m in ["mape", "smape"] else ""
    print(f"  {m.upper():<8} {baseline_val:>14.4f}{sufijo} {lgbm_val:>14.4f}{sufijo} {mejora_abs:>+14.4f} {mejora_pct:>+8.2f}%")

print("\nTEST 2026 (out of sample):")
for m in ["mae", "rmse", "mape", "smape"]:
    baseline_val = float(base_t[m])
    lgbm_val = metricas_test[m]
    mejora_abs = baseline_val - lgbm_val
    mejora_pct = (mejora_abs / baseline_val * 100) if baseline_val != 0 else 0
    sufijo = "%" if m in ["mape", "smape"] else ""
    print(f"  {m.upper():<8} {baseline_val:>14.4f}{sufijo} {lgbm_val:>14.4f}{sufijo} {mejora_abs:>+14.4f} {mejora_pct:>+8.2f}%")

print("=" * 75)
if metricas_valid["mae"] < float(base_v["mae"]) and metricas_test["mae"] < float(base_t["mae"]):
    print("OK: LightGBM mejora al baseline en MAE en ambos conjuntos.")
else:
    print("ATENCION: LightGBM no mejora al baseline en MAE. Revisar.")

## 9. Agregar predicciones al DataFrame para análisis

In [ ]:
valid["y_pred_lgbm"] = y_pred_valid
test["y_pred_lgbm"] = y_pred_test

# Mergear baseline para tener todo junto
valid_full = valid.merge(
    baseline_preds[["item", "centro_operacion", "fecha_inicio_semana", "pred_baseline"]],
    on=["item", "centro_operacion", "fecha_inicio_semana"],
    how="left",
)
test_full = test.merge(
    baseline_preds[["item", "centro_operacion", "fecha_inicio_semana", "pred_baseline"]],
    on=["item", "centro_operacion", "fecha_inicio_semana"],
    how="left",
)

# Agregar clasificacion
for col in ["clase_abc", "segmento_abc_xyz"]:
    if col in valid_full.columns:
        valid_full = valid_full.drop(columns=[col])
    if col in test_full.columns:
        test_full = test_full.drop(columns=[col])

valid_full = valid_full.merge(clasif, on="item", how="left")
test_full = test_full.merge(clasif, on="item", how="left")

valid_full["clase_abc"] = valid_full["clase_abc"].fillna("C")
valid_full["segmento_abc_xyz"] = valid_full["segmento_abc_xyz"].fillna("CZ")
test_full["clase_abc"] = test_full["clase_abc"].fillna("C")
test_full["segmento_abc_xyz"] = test_full["segmento_abc_xyz"].fillna("CZ")

print(f"valid_full: {len(valid_full):,} filas")
print(f"test_full:  {len(test_full):,} filas")

## 10. Métricas por segmento ABC × XYZ

¿LightGBM mejora más en algunos segmentos que en otros?

In [ ]:
def matriz_metrica(df, columna_pred="y_pred_lgbm", metrica="mae"):
    resultados = []
    for seg, sub in df.groupby("segmento_abc_xyz", observed=True):
        m = calcular_metricas(sub["cantidad_total"], sub[columna_pred])
        clase_abc = seg[0]
        clase_xyz = seg[1]
        resultados.append({
            "clase_abc": clase_abc, "clase_xyz": clase_xyz,
            "metrica": m[metrica], "n_obs": m["n_obs"],
        })
    df_res = pd.DataFrame(resultados)
    matriz = df_res.pivot(index="clase_abc", columns="clase_xyz", values="metrica")
    return matriz.reindex(index=["A", "B", "C"], columns=["X", "Y", "Z"])

# MAE de LightGBM por segmento
mae_lgbm_valid = matriz_metrica(valid_full, "y_pred_lgbm", "mae")
mae_lgbm_test = matriz_metrica(test_full, "y_pred_lgbm", "mae")

# MAE del baseline por segmento (para comparar)
mae_base_valid = matriz_metrica(valid_full, "pred_baseline", "mae")
mae_base_test = matriz_metrica(test_full, "pred_baseline", "mae")

print("MAE de LightGBM en VALIDATION por segmento:")
print(mae_lgbm_valid.round(2))
print("\nMAE de LightGBM en TEST 2026 por segmento:")
print(mae_lgbm_test.round(2))

In [ ]:
# Mejora relativa = (baseline - lgbm) / baseline * 100
mejora_valid = ((mae_base_valid - mae_lgbm_valid) / mae_base_valid * 100).round(2)
mejora_test = ((mae_base_test - mae_lgbm_test) / mae_base_test * 100).round(2)

print("Mejora porcentual de LightGBM sobre baseline en VALIDATION (% de reduccion de MAE):")
print(mejora_valid)
print("\nMejora porcentual en TEST 2026:")
print(mejora_test)
print("\nInterpretacion: valores POSITIVOS = LightGBM mejor que baseline.")
print("Valores NEGATIVOS = LightGBM peor que baseline (esos segmentos hay que revisar).")

In [ ]:
# Heatmaps comparativos
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Fila 1: MAE de LightGBM
sns.heatmap(mae_lgbm_valid, annot=True, fmt=".2f", cmap="RdYlGn_r",
            ax=axes[0, 0], cbar_kws={"label": "MAE"}, linewidths=1)
axes[0, 0].set_title("LightGBM — MAE en VALIDATION", pad=10)
axes[0, 0].set_xlabel("Clase XYZ")
axes[0, 0].set_ylabel("Clase ABC")

sns.heatmap(mae_lgbm_test, annot=True, fmt=".2f", cmap="RdYlGn_r",
            ax=axes[0, 1], cbar_kws={"label": "MAE"}, linewidths=1)
axes[0, 1].set_title("LightGBM — MAE en TEST 2026", pad=10)
axes[0, 1].set_xlabel("Clase XYZ")
axes[0, 1].set_ylabel("Clase ABC")

# Fila 2: Mejora vs baseline
sns.heatmap(mejora_valid, annot=True, fmt=".1f", cmap="RdYlGn",
            ax=axes[1, 0], cbar_kws={"label": "% mejora"}, linewidths=1, center=0)
axes[1, 0].set_title("Mejora % sobre baseline (VALIDATION)", pad=10)
axes[1, 0].set_xlabel("Clase XYZ")
axes[1, 0].set_ylabel("Clase ABC")

sns.heatmap(mejora_test, annot=True, fmt=".1f", cmap="RdYlGn",
            ax=axes[1, 1], cbar_kws={"label": "% mejora"}, linewidths=1, center=0)
axes[1, 1].set_title("Mejora % sobre baseline (TEST 2026)", pad=10)
axes[1, 1].set_xlabel("Clase XYZ")
axes[1, 1].set_ylabel("Clase ABC")

plt.tight_layout()
guardar(fig, "01_comparacion_lightgbm_vs_baseline")
plt.show()

## 11. Feature importance

¿Qué features están realmente aportando al modelo?

In [ ]:
# Feature importance (gain = ganancia total que aporta la feature)
fi = pd.DataFrame({
    "feature": modelo.feature_name(),
    "importance_gain": modelo.feature_importance(importance_type="gain"),
    "importance_split": modelo.feature_importance(importance_type="split"),
}).sort_values("importance_gain", ascending=False).reset_index(drop=True)

print("Top 15 features por importancia (gain):")
print(fi.head(15))

In [ ]:
# Visualizar feature importance
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

top_fi = fi.head(20)

# Por gain
axes[0].barh(top_fi["feature"][::-1], top_fi["importance_gain"][::-1] / 1e6,
             color=COLOR_LIGHTGBM, edgecolor="white")
axes[0].set_title("Top 20 features por GAIN", pad=10)
axes[0].set_xlabel("Gain (millones)")

# Por split
top_fi_split = fi.sort_values("importance_split", ascending=False).head(20)
axes[1].barh(top_fi_split["feature"][::-1], top_fi_split["importance_split"][::-1],
             color=COLOR_PRIMARIO, edgecolor="white")
axes[1].set_title("Top 20 features por SPLITS (# usos)", pad=10)
axes[1].set_xlabel("Numero de splits")

plt.tight_layout()
guardar(fig, "02_feature_importance")
plt.show()

print("\nInterpretacion:")
print("  - GAIN: cuanta reduccion de error aporta la feature.")
print("  - SPLITS: cuantas veces se uso para hacer cortes en los arboles.")

## 12. Real vs Predicho (mismos SKUs del baseline para comparar)

In [ ]:
# Top 4 SKUs clase A en centro 001 (los mismos del notebook 05)
top_skus = (
    valid_full[(valid_full["clase_abc"] == "A") & (valid_full["centro_operacion"] == "001")]
    .groupby("item")["cantidad_total"].sum()
    .sort_values(ascending=False)
    .head(4)
    .index.tolist()
)

# Combinar dataset completo con predicciones
combined = pd.concat([
    valid_full[["item", "centro_operacion", "fecha_inicio_semana", "cantidad_total",
                "y_pred_lgbm", "pred_baseline"]],
    test_full[["item", "centro_operacion", "fecha_inicio_semana", "cantidad_total",
               "y_pred_lgbm", "pred_baseline"]],
]).sort_values(["item", "centro_operacion", "fecha_inicio_semana"])

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for i, sku in enumerate(top_skus):
    ax = axes[i]
    sub = combined[(combined["item"] == sku) & (combined["centro_operacion"] == "001")]

    ax.plot(sub["fecha_inicio_semana"], sub["cantidad_total"],
            label="Real", color=COLOR_PRIMARIO, linewidth=1.5)
    ax.plot(sub["fecha_inicio_semana"], sub["pred_baseline"],
            label="Baseline (MA-4)", color=COLOR_SECUNDARIO, linewidth=1.3, alpha=0.7)
    ax.plot(sub["fecha_inicio_semana"], sub["y_pred_lgbm"],
            label="LightGBM", color=COLOR_LIGHTGBM, linewidth=1.5, alpha=0.9)

    ax.axvline(pd.Timestamp("2026-01-01"), color=COLOR_ROJO, linestyle=":",
                linewidth=1.5, label="Inicio test 2026")

    ax.set_title(f"SKU {sku} en centro 001", pad=10)
    ax.set_xlabel("Semana")
    ax.set_ylabel("Cantidad")
    ax.legend(loc="upper left", fontsize=8)
    ax.tick_params(axis="x", rotation=45)

plt.suptitle("Real vs Baseline vs LightGBM para top 4 SKUs clase A en centro 001",
             y=1.01, fontsize=14)
plt.tight_layout()
guardar(fig, "03_real_vs_pred_top_skus")
plt.show()

## 13. Análisis: ¿dónde mejora LightGBM al baseline?

In [ ]:
# Para cada observacion en valid, calcular cual modelo es mejor
valid_full["err_baseline"] = np.abs(valid_full["cantidad_total"] - valid_full["pred_baseline"])
valid_full["err_lgbm"] = np.abs(valid_full["cantidad_total"] - valid_full["y_pred_lgbm"])
valid_full["lgbm_mejor"] = valid_full["err_lgbm"] < valid_full["err_baseline"]

pct_mejor_global = valid_full["lgbm_mejor"].mean() * 100
print(f"En el {pct_mejor_global:.2f}% de las observaciones de validation, LightGBM tiene MENOR error que el baseline.")
print()

# Por segmento
print("Porcentaje de observaciones donde LightGBM gana al baseline (por segmento ABC):")
gana_por_abc = valid_full.groupby("clase_abc")["lgbm_mejor"].agg(["mean", "count"])
gana_por_abc["mean"] = gana_por_abc["mean"] * 100
gana_por_abc.columns = ["pct_lgbm_gana", "n_obs"]
print(gana_por_abc.round(2))

## 14. Guardado de modelo, predicciones y métricas

In [ ]:
# Guardar modelo entrenado
ruta_modelo = ROOT / "data" / "processed" / "lightgbm_model.txt"
modelo.save_model(str(ruta_modelo), num_iteration=modelo.best_iteration)
print(f"Modelo guardado: {ruta_modelo.relative_to(ROOT)} ({ruta_modelo.stat().st_size / 1024:.1f} KB)")

In [ ]:
# Guardar predicciones en MySQL
pronosticos_lgbm = pd.concat([
    valid_full[["item", "centro_operacion", "fecha_inicio_semana",
                "cantidad_total", "y_pred_lgbm"]]
    .rename(columns={"cantidad_total": "cantidad_real", "y_pred_lgbm": "cantidad_predicha"})
    .assign(split="valid", modelo="lightgbm"),
    test_full[["item", "centro_operacion", "fecha_inicio_semana",
               "cantidad_total", "y_pred_lgbm"]]
    .rename(columns={"cantidad_total": "cantidad_real", "y_pred_lgbm": "cantidad_predicha"})
    .assign(split="test_2026", modelo="lightgbm"),
])

# Borrar predicciones anteriores de lightgbm si existen
with engine.begin() as conn:
    conn.execute(text("DELETE FROM pronosticos WHERE modelo = 'lightgbm'"))

print(f"Guardando {len(pronosticos_lgbm):,} predicciones de LightGBM en chunks de 1000...")
pronosticos_lgbm.to_sql(
    "pronosticos", engine, if_exists="append", index=False, chunksize=1000,
)

with engine.connect() as conn:
    n = conn.execute(text("SELECT COUNT(*) FROM pronosticos WHERE modelo='lightgbm'")).scalar()
print(f"Predicciones LightGBM guardadas: {n:,} filas")

In [ ]:
# Guardar metricas
metricas_resumen = pd.DataFrame([
    {"modelo": "lightgbm", "split": "valid", "segmento": "global",
     "mae": metricas_valid["mae"], "rmse": metricas_valid["rmse"],
     "mape": metricas_valid["mape"], "smape": metricas_valid["smape"],
     "n_obs": metricas_valid["n_obs"]},
    {"modelo": "lightgbm", "split": "test_2026", "segmento": "global",
     "mae": metricas_test["mae"], "rmse": metricas_test["rmse"],
     "mape": metricas_test["mape"], "smape": metricas_test["smape"],
     "n_obs": metricas_test["n_obs"]},
])

# Por clase ABC
for clase in ["A", "B", "C"]:
    sub_v = valid_full[valid_full["clase_abc"] == clase]
    sub_t = test_full[test_full["clase_abc"] == clase]
    if len(sub_v) > 0:
        mv = calcular_metricas(sub_v["cantidad_total"], sub_v["y_pred_lgbm"])
        metricas_resumen = pd.concat([metricas_resumen, pd.DataFrame([{
            "modelo": "lightgbm", "split": "valid", "segmento": f"abc_{clase}",
            "mae": mv["mae"], "rmse": mv["rmse"], "mape": mv["mape"], "smape": mv["smape"],
            "n_obs": mv["n_obs"],
        }])])
    if len(sub_t) > 0:
        mt = calcular_metricas(sub_t["cantidad_total"], sub_t["y_pred_lgbm"])
        metricas_resumen = pd.concat([metricas_resumen, pd.DataFrame([{
            "modelo": "lightgbm", "split": "test_2026", "segmento": f"abc_{clase}",
            "mae": mt["mae"], "rmse": mt["rmse"], "mape": mt["mape"], "smape": mt["smape"],
            "n_obs": mt["n_obs"],
        }])])

metricas_resumen = metricas_resumen.reset_index(drop=True)

# Borrar metricas anteriores de lightgbm
with engine.begin() as conn:
    conn.execute(text("DELETE FROM metricas_modelos WHERE modelo = 'lightgbm'"))

metricas_resumen.to_sql("metricas_modelos", engine, if_exists="append", index=False)
print(f"Metricas guardadas: {len(metricas_resumen)} filas en tabla metricas_modelos.")
metricas_resumen

## 15. Resumen y conclusiones

### 15.1 Resultados clave

Ejecuta la siguiente celda para ver el resumen ejecutivo del notebook.

In [ ]:
print("=" * 75)
print("LIGHTGBM — RESUMEN FINAL")
print("=" * 75)
print(f"\nMejor iteracion: {modelo.best_iteration}")
print(f"\n{'Metrica':<10} {'Baseline':>12} {'LightGBM':>12} {'Mejora':>15}")
print("-" * 75)
print("VALIDATION:")
for m in ["mae", "rmse", "mape", "smape"]:
    bv = float(base_v[m])
    lv = metricas_valid[m]
    mejora = (bv - lv) / bv * 100
    sufijo = "%" if m in ["mape", "smape"] else ""
    print(f"  {m.upper():<8} {bv:>11.2f}{sufijo} {lv:>11.2f}{sufijo} {mejora:>+13.2f}%")
print("\nTEST 2026:")
for m in ["mae", "rmse", "mape", "smape"]:
    bt = float(base_t[m])
    lt = metricas_test[m]
    mejora = (bt - lt) / bt * 100
    sufijo = "%" if m in ["mape", "smape"] else ""
    print(f"  {m.upper():<8} {bt:>11.2f}{sufijo} {lt:>11.2f}{sufijo} {mejora:>+13.2f}%")

print(f"\nLightGBM gana al baseline en el {pct_mejor_global:.2f}% de observaciones (validation).")
print()
print(f"Archivos generados:")
print(f"  - Modelo: data/processed/lightgbm_model.txt")
print(f"  - Predicciones MySQL: tabla pronosticos (modelo='lightgbm')")
print(f"  - Metricas MySQL: tabla metricas_modelos (modelo='lightgbm')")
print(f"  - Figuras: reports/figures/lightgbm/")
print(f"\nSiguiente paso: notebook 07 (XGBoost) para comparacion.")
print("=" * 75)

### 15.2 Lectura para el informe (Capítulo 5)

Estos resultados son insumos directos para el Capítulo 5 (OE3) del informe técnico:

- **El MAE de LightGBM** versus el del baseline mide el aporte real del modelo.
- **El feature importance** justifica qué variables son relevantes para predecir la demanda en ConstruNorte.
- **La comparación por segmento ABC** muestra dónde aporta más valor el modelo.

### 15.3 Próximos pasos

- **Notebook 07 — XGBoost:** modelo de comparación con un algoritmo similar pero distinto.
- **Notebook 08 — Prophet:** modelo univariante para los top 50 SKUs clase A.
- **Notebook 09 — Evaluación final:** comparación lado a lado de los 4 modelos.